# Deep Learning Based Early Warning System for Water-Borne Disease Prediction

**Academic Mini Project** | Complete Training Notebook

This notebook demonstrates the complete training pipeline:
1. Dataset Generation & Exploration
2. Data Preprocessing
3. Deep Neural Network Training
4. Random Forest & XGBoost Comparison
5. SHAP Explainability
6. PDF Report Generation


In [ ]:
import os, sys
# Go to project root
os.chdir('..')
print('Working directory:', os.getcwd())

## 1. Dataset Generation

In [ ]:
sys.path.insert(0, 'dataset')
from generate_dataset import generate_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = generate_dataset()
print(f'Dataset shape: {df.shape}')
print('\nClass distribution:')
print(df['disease_risk'].value_counts())
df.head()

In [ ]:
# Dataset statistics
df.describe().round(2)

In [ ]:
# Risk distribution pie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dist = df['disease_risk'].value_counts()
colors = ['#4caf50', '#ff9800', '#f44336']
axes[0].pie(dist.values, labels=dist.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Disease Risk Distribution', fontsize=14, fontweight='bold')

# Feature correlation heatmap (numerical only)
num_cols = ['ph', 'turbidity', 'dissolved_oxygen', 'temperature', 'rainfall',
            'humidity', 'population_density', 'sanitation_index',
            'bacterial_count', 'ecoli_count', 'chlorine_level', 'previous_cases']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, ax=axes[1],
            linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

## 2. Run Full Training Pipeline

The cell below runs the complete `train_model.py` script which handles all preprocessing, model training, evaluation, SHAP analysis and PDF report generation.

In [ ]:
# Run the full pipeline
import subprocess
result = subprocess.run(
    [sys.executable, 'train_model.py'],
    capture_output=False,
    text=True
)
print('Return code:', result.returncode)

## 3. Load and Inspect Results

In [ ]:
# Show accuracy report
import pandas as pd
acc_df = pd.read_csv('reports/accuracy_report.csv')
acc_df['Accuracy %'] = (acc_df['Accuracy'] * 100).round(2)
print('\n=== Model Accuracy Comparison ===')
print(acc_df.to_string(index=False))

In [ ]:
# Show generated reports
from IPython.display import Image, display
print('Confusion Matrix:')
display(Image('reports/confusion_matrix.png', width=500))
print('\nTraining History:')
display(Image('reports/training_history.png', width=700))
print('\nSHAP Feature Importance:')
display(Image('reports/shap_analysis.png', width=700))

## 4. Make a Prediction

In [ ]:
import pickle, numpy as np
import tensorflow as tf

# Load artifacts
model    = tf.keras.models.load_model('models/waterborne_model.h5')
scaler   = pickle.load(open('models/scaler.pkl', 'rb'))
le       = pickle.load(open('models/label_encoder.pkl', 'rb'))
feat_col = pickle.load(open('models/feature_columns.pkl', 'rb'))

print('Model loaded. Classes:', le.classes_)
print('Features:', len(feat_col))

In [ ]:
# Example: HIGH RISK scenario
sample = {
    'ph': 5.2, 'turbidity': 80, 'dissolved_oxygen': 3,
    'temperature': 36, 'rainfall': 400, 'humidity': 85,
    'population_density': 6000, 'sanitation_index': 20,
    'bacterial_count': 2000, 'ecoli_count': 200,
    'chlorine_level': 0.05, 'previous_cases': 60,
    'water_source_Pond': 1, 'region_type_Rural': 1, 'season_Monsoon': 1,
}

row = [sample.get(col, 0.0) for col in feat_col]
X   = np.array([row], dtype=np.float32)
X_s = scaler.transform(X)
probs = model.predict(X_s, verbose=0)[0]
idx   = int(np.argmax(probs))

print(f'Predicted Risk: {le.classes_[idx]}')
print(f'Confidence:     {probs[idx]*100:.1f}%')
for i, cls in enumerate(le.classes_):
    print(f'  {cls}: {probs[i]*100:.1f}%')

## 5. PDF Report

The PDF report has been automatically generated and saved to `reports/final_report.pdf`.

In [ ]:
import os
pdf_path = 'reports/final_report.pdf'
if os.path.exists(pdf_path):
    size_kb = os.path.getsize(pdf_path) / 1024
    print(f'✅ PDF Report: {pdf_path} ({size_kb:.1f} KB)')
else:
    print('⚠️  PDF not found. Run train_model.py first.')

print('\n✅ Project pipeline complete!')
print('   Backend:  uvicorn app:app --port 8000  (from backend/)')
print('   Frontend: npm run dev                  (from frontend/)')